Which values of $\alpha$ and $\beta$ are consistent with financial data?

- $\beta \in [0.8,1]$
- $\alpha \in [0,0.2]$
- $\phi = \alpha + \beta \geq 0.98$
- The condition in the paper, $ 1 - \alpha^2 \kappa - 2 \alpha \beta - \beta^2 > 0$ is only satisfied 20% of the time, and when it is, it is too close to zero, leading to numeric instability.

In [ ]:
import time, os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import yfinance as yf
import scipy
import ray
from dataclasses import asdict
from tqdm.auto import tqdm
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch
from curl_cffi import requests as curl_requests

from functions import estimate_parameters, plot_parameters_all, round_p_value, hplot

In [ ]:
RECOMPUTE = False

In [ ]:
# Session that skips SSL verification (e.g. for corporate proxies with self-signed certs)
_session = curl_requests.Session(impersonate="chrome", verify=False)

os.makedirs("cache", exist_ok=True)

In [ ]:
ids = {
    'sp500':    "^GSPC",
    'nasdaq':   "^IXIC",
    'ftse':     "^FTSE",
    'dax':      "^GDAXI",
    'cac':      "^FCHI",
    'euronext': "^N100",
    'hsi':      "^HSI",
    'sti':      "^STI",
    'au':       "^AXJO",
    'n225':     "^N225",
    'kospi':    "^KS11",
    'sensex':   "^BSESN",
    'Information Technology': 'XLK',
    'Health Care':            'XLV',
    'Consumer Discretionary': 'XLY',
    'Communication Services': 'XLC',
    'Financials':             'XLF',
    'Industrials':            'XLI',
    'Consumer Staples':       'XLP',
    'Utilities':              'XLU',
    'Materials':              'XLB',
    'Real Estate':            'XLRE',
    'Energy':                 'XLE',
} | {
    'sp500':    "^GSPC",    # US
    'nasdaq':   "^IXIC",    # US
    'r2000':    '^RUT',     # US
    'dji':      '^DJI',     # US
    'ftse':     "^FTSE",    # UK
    'dax':      "^GDAXI",   # DE
    'cac':      "^FCHI",    # FR
    'euronext': "^N100",    # EU
    'hsi':      "^HSI",     # HK
    'sti':      "^STI",     # SG
    'au':       "^AXJO",    # AU
    'n225':     "^N225",    # JP
    'kospi':    "^KS11",    # KR
    'sensex':   "^BSESN",   # IN
    'taiwan':   "^TWII",
    'world':    "XWD.TO",   # That is an ETF
    'my':       "^KLSE",
    'ph':       "PSEI.PS",
    'id':       "^JKSE",
    'bel':      "^BFX",
    'ru':       "IMOEX.ME",
    'br':       "^BVSP",
    'mx':       "^MXX",

    'Information Technology': 'XLK',
    'Health Care':            'XLV',
    'Consumer Discretionary': 'XLY',
    'Communication Services': 'XLC',
    'Financials':             'XLF',
    'Industrials':            'XLI',
    'Consumer Staples':       'XLP',
    'Utilities':              'XLU',
    'Materials':              'XLB',
    'Real Estate':            'XLRE',
    'Energy':                 'XLE',

    "vix": "^VIX",

    'Apple':     'AAPL',
    'Microsoft': 'MSFT',
    'Amazon':    'AMZN',
    'Google':    'GOOG',
    'Facebook':  'FB',
    'Tesla':     'TSLA',
    'Berkshire': 'BRK-B',
    'Visa':      'V',
    'NVidia':    'NVDA',


    'Crude Oil':     'CL=F',
    'Gold':          'GC=F',
    'Silver':        'SI=F',
    'Platinum':      'PL=F',
    'Copper':        'HG=F',
    'Palladium':     'PA=F',
    'Heating oil':   'HO=F',
    'Natural gas':   'NG=F',
    'RBOB gasoline': 'RB=F',
    'Brent':         'BZ=F',
    'Corn':          'ZC=F',
    'Oat':           'ZO=F',
    'HRW wheat':     'KE=F',
    'Rice':          'ZR=F',
    'Soybean meal':  'ZM=F',
    'Soybean oil':   'ZL=F',
    'Soybean':       'ZS=F',
    'Feeder cattle': 'GF=F',
    'Lean hogs':     'HE=F',
    'Live cattle':   'LE=F',
    'Cocoa':         'CC=F',
    'Cotton':        'CT=F',
    'Orange juice':  'OJ=F',
    'Sugar':         'SB=F',

    '10y':       '^TNX',

    'BTC':       'BTC-USD',  # Crypto
    'CMC200':    '^CMC200',  # Crypto
}

In [ ]:
filename = "cache/yfinance_returns.parquet"

if not os.path.exists(filename):
    d = {}
    for label, id in ids.items():
        if label in d:
            continue
        if np.random.uniform() < .1:
            time.sleep(5)
        print( f"{label}: {id}" )
        d[id] = yf.Ticker(id, session=_session).history(period="max")['Close']

    pd.DataFrame(d).to_parquet(filename)

data = pd.read_parquet(filename)

In [ ]:
fig, axs = plt.subplots( data.shape[1], 1, figsize = (10, data.shape[1]), layout = 'constrained', dpi = 100 )
for ax, id in zip( axs, tqdm(data.columns) ):
    y = np.log(data[id]).dropna().diff().dropna()
    model = arch_model(y, vol='Garch', p=1, q=1, dist='skewt', mean = 'Zero', rescale=True)
    res = model.fit(disp="off")
    nlags = 5
    arch_lm_stat, arch_lm_pvalue, f_stat, f_pvalue = het_arch(y - y.mean(), nlags=nlags)
    s = 3
    ax.fill_between( 
        res.conditional_volatility.index,
        - s * res.conditional_volatility / res.scale,
        s * res.conditional_volatility / res.scale,
        color = 'tab:orange',
        alpha = .2,
    )
    hplot( y, ax = ax )
    ax.axis('off')
    ax.set_xlim( y.index[0], y.index[-1] )

    text = (
        f"α={res.params['alpha[1]']:.2f} "
        f"β={res.params['beta[1]']:.2f} "
        f"p={round_p_value(arch_lm_pvalue)}"
    )
    text = ax.text( 
        0, 1, 
        text,
        ha = 'left', va = 'top', 
        transform = ax.transAxes,
    )
    text.set_path_effects([
        path_effects.Stroke(linewidth=4, foreground='white', alpha = .5),
        path_effects.Normal()
    ])
    
    ax.text( 
        1, 1, 
        { v: k for k, v in ids.items() }[id], 
        ha = 'right', va = 'top', 
        transform = ax.transAxes,
    )
plt.show()

In [ ]:
filename = "cache/parameters_yfinance.parquet"

@ray.remote
def estimate_parameters_remote( y ):
    return estimate_parameters( y )

if os.path.exists(filename) and not RECOMPUTE:
    parameters = pd.read_parquet(filename)
else:
    parameters = {}

    for id in tqdm(data.columns):
        y = data[id].dropna()
        y = np.log( y ).diff().dropna()
        y = y[ np.isfinite(y) ]
        parameters[id] = estimate_parameters_remote.remote( y )

    parameters = { k: ray.get(v) for k, v in tqdm(parameters.items()) }
    parameters = { k: asdict(v) for k, v in parameters.items() }
    parameters = pd.DataFrame( parameters ).T
    parameters.to_parquet(filename)

In [ ]:
parameters.T

In [ ]:
plot_parameters_all( parameters )

In [ ]:
print( 
    f"Only {int(100 * np.mean( parameters['denominator'] > 0 ))}% "
    "of time series satisfy the condition 1-α²κ-2αβ-β²>0" 
)